# Experiment 4 — RC4 Stream Cipher
KSA (Key Scheduling Algorithm) + PRGA (Pseudo-Random Generation Algorithm)

## KSA — Key Scheduling Algorithm

In [ ]:
def ksa(key):
    S=list(range(256)); j=0
    for i in range(256):
        j=(j+S[i]+key[i%len(key)])%256
        S[i],S[j]=S[j],S[i]
    return S

key = b'SecretKey'
S   = ksa(key)
print(f"Key      : {key}")
print(f"S[0..7]  : {S[:8]}")
print("✓ KSA complete")


## PRGA — Pseudo-Random Generation Algorithm

In [ ]:
def prga(S, length):
    S=S[:]; i=j=0; stream=[]
    for _ in range(length):
        i=(i+1)%256; j=(j+S[i])%256
        S[i],S[j]=S[j],S[i]
        stream.append(S[(S[i]+S[j])%256])
    return stream

stream = prga(ksa(key), 16)
print(f"Keystream (16 bytes) : {' '.join(f'{b:02X}' for b in stream)}")


## RC4 Encrypt / Decrypt

In [ ]:
def rc4(key, data):
    S=ksa(key); stream=prga(S,len(data))
    return bytes(b^k for b,k in zip(data,stream))

pt = b'Hello RC4 World!'
ct = rc4(key, pt)
dt = rc4(key, ct)

print(f"Plaintext  : {pt}")
print(f"Ciphertext : {ct.hex().upper()}")
print(f"Decrypted  : {dt}")
assert dt == pt
print("✓ Roundtrip verified")


## Keystream Statistics

In [ ]:
full_stream = prga(ksa(key), 256)
unique = len(set(full_stream))
mean   = sum(full_stream)/len(full_stream)
print(f"Unique byte values : {unique}/256")
print(f"Mean value         : {mean:.2f}  (expected ≈ 127.5)")


## Key Sensitivity

In [ ]:
key2 = b'SecretKey!'
ct2  = rc4(key2, pt)
diff = sum(a!=b for a,b in zip(ct, ct2))
print(f"Key 1 CT : {ct.hex().upper()}")
print(f"Key 2 CT : {ct2.hex().upper()}")
print(f"Different bytes : {diff}/{len(pt)}")


## Known Weakness — Keystream Reuse

In [ ]:
p1 = b'ATTACK AT DAWN!!'
p2 = b'ATTACK AT NOON!!'
c1 = rc4(key, p1)
c2 = rc4(key, p2)
xored = bytes(a^b for a,b in zip(c1,c2))
recovered = bytes(a^b for a,b in zip(xored,p1))
print(f"CT1 XOR CT2 XOR PT1 = {recovered}")
assert recovered == p2
print("⚠ Same key → keystream reuse reveals PT2!")
print("✓ Never reuse RC4 key+nonce")
